# Student Performance Analysis – Step by Step
The college wants to understand what drives first-semester marks so that mentors can support at-risk students early.

**Questions to answer**
1. What is the overall performance of the batch?
2. Do attendance and study hours affect marks?
3. Does performance differ by branch, gender, residence or tutoring support?
4. Which students are at risk (average marks below 50)?

The dataset is synthetic (computer-generated). Student names are fictional, and it contains no real student data.

## Step 1 – Import libraries
Import pandas and NumPy for data work, and matplotlib and seaborn for charts.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display   # shows tables below the cell

sns.set_theme(style="whitegrid")

# Standalone setup: everything is created next to this notebook (no zip / folder structure needed)
BASE = Path("student_performance_project")
(BASE / "data/raw").mkdir(parents=True, exist_ok=True)
(BASE / "data/processed").mkdir(parents=True, exist_ok=True)
(BASE / "images").mkdir(parents=True, exist_ok=True)
print("Project folder:", BASE.resolve())


### Embedded raw dataset
The raw CSV is stored inside this notebook, so no separate file is needed. It is saved to `data/raw/` the first time you run it.

In [ ]:
RAW_CSV = r"""Student_ID,Student_Name,Gender,Branch,Residence,Attendance_Pct,Study_Hours_Per_Week,Sleep_Hours,Tutoring,Maths_Marks,Physics_Marks,Programming_Marks
S1133,Varun Bansal,Male,IT,Day Scholar,62.5,14.1,6.6,No,52.0,60.0,58.0
S1248,Kabir Kapoor,Male,IT,Day Scholar,67.8,8.8,5.4,No,42.0,51.0,65.0
S1225,Ananya Jain,Female,CSE,Day Scholar,80.5,13.5,7.0,No,64.0,58.0,65.0
S1025,Karan Singh,Male, cse ,Hostel,70.1,10.6,8.1,Yes,38.0,62.0,60.0
S1058,Sneha Bansal,Female,IT,Day Scholar,79.7,12.7,8.0,Yes,54.0,56.0,63.0
S1066,Aryan Srivastava,Male,IT,hostel,75.8,2.2,6.8,Yes,53.0,44.0,57.0
S1003,Nisha Singh,Female,ME,Hostel,,16.9,9.0,Yes,67.0,47.0,75.0
S1180,Diya Gupta,Female,ECE,Hostel,79.9,10.5,6.6,Yes,45.0,60.0,67.0
S1210,Nikhil Sharma,Male,IT,hostel,76.3,14.5,6.2,No,53.0,68.0,36.0
S1095,Varun Sharma,Male,CSE,Day Scholar,73.9,13.5,4.0,No,56.0,46.0,55.0
S1242,Isha Tiwari,Female,ECE,Day Scholar,76.3,13.6,6.7,No,60.0,66.0,55.0
S1170,Siddharth Srivastava,Male,ME,Hostel,69.0,15.2,7.7,Yes,53.0,66.0,65.0
S1004,Meera Singh,Female,ECE,Hostel,100.0,18.3,7.1,No,70.0,69.0,80.0
S1097,Priya Joshi,male,IT,Day Scholar,85.4,11.4,7.9,No,,50.0,65.0
S1201,Priya Srivastava,Female,ECE,Day Scholar,76.2,3.5,6.7,Yes,,43.0,49.0
S1023,Aarav Jain,Male,ME,Hostel,85.0,12.3,7.2,No,61.0,59.0,59.0
S1127,Dev Yadav,Male,CSE,Day Scholar,75.4,17.3,5.0,Yes,51.0,72.0,54.0
S1075,Vivaan Pandey,Male,CSE,Day Scholar,81.4,19.2,7.8,Yes,67.0,78.0,56.0
S1089,Khushi Saxena,Female,CSE,Hostel,80.4,15.4,5.9,No,49.0,56.0,65.0
S1079,Sakshi Verma,Female,IT,Hostel,64.6,4.4,6.6,No,27.0,52.0,37.0
S1154,Muskan Yadav,Female,IT,Day Scholar,84.2,10.4,7.1,No,49.0,58.0,44.0
S1134,Rohan Rastogi,Male,CSE,Day Scholar,88.3,14.9,8.9,Yes,55.0,49.0,70.0
S1063,Pranav Pandey,Male,IT,Day Scholar,99.7,12.4,7.5,No,46.0,52.0,71.0
S1123,Kabir Yadav,Male,CSE,Day Scholar,58.6,6.5,6.8,No,43.0,48.0,47.0
S1100,Tanya Saxena,Female,ME,Hostel,80.9,11.0,5.4,Yes,45.0,48.0,53.0
S1121,Kabir Agarwal,Male,IT,Day Scholar,73.7,9.1,7.3,No,48.0,49.0,70.0
S1161,Tanmay Kapoor,Male,CSE,Hostel,82.6,13.2,7.1,No,51.0,61.0,40.0
S1217,Pooja Agarwal,Female,CSE,Hostel,80.3,10.6,6.7,yes,61.0,45.0,60.0
S1112,Tanmay Mishra,Male,CSE,Day Scholar,66.5,9.4,8.3,Yes,53.0,55.0,66.0
S1051,Riya Sharma,Female,CSE,Day Scholar,75.5,9.8,7.3,No,,33.0,59.0
S1012,Meera Yadav,Female,ME,Day Scholar,78.3,7.6,8.7,No,46.0,55.0,51.0
S1205,Vivaan Verma,Male,CSE,Hostel,81.5,9.4,6.0,Yes,54.0,62.0,54.0
S1216,Sakshi Pandey,Female,ECE,Day Scholar,76.4,1.0,7.6,Yes,45.0,42.0,62.0
S1222,Kavya Rastogi,Female,IT,Day Scholar,67.3,14.3,5.9,No,54.0,52.0,62.0
S1081,Varun Mishra,Male,ECE,Hostel,68.1,8.0,8.2,yes,50.0,58.0,40.0
S1071,Pooja Chauhan,Female,CSE,Day Scholar,67.5,6.0,8.1,No,60.0,20.0,55.0
S1158,Isha Pandey,Female,ECE,hostel,74.3,15.9,6.2,No,50.0,55.0,52.0
S1188,Neha Tiwari,Female,CSE,Day Scholar,89.3,16.8,7.4,No,70.0,76.0,78.0
S1220,Diya Tiwari,Female,IT,Hostel,69.5,9.3,6.0,Yes,55.0,57.0,69.0
S1053,Aditi Chauhan,Female,ECE,Day Scholar,87.6,14.8,8.7,No,60.0,56.0,58.0
S1110,Tanya Agarwal,Female,ME,Hostel,80.7,13.7,6.7,Yes,62.0,46.0,66.0
S1008,Karan Yadav,Male,ECE,Day Scholar,70.7,16.5,6.5,Yes,60.0,57.0,68.0
S1061,Riya Chauhan,Female,ECE,Hostel,94.1,9.8,6.3,Yes,57.0,61.0,55.0
S1092,Ananya Pandey,Female,ECE,Day Scholar,78.7,12.0,7.0,No,48.0,44.0,43.0
S1091,Aditi Gupta,Female,CSE,Day Scholar,94.8,14.5,5.2,Yes,76.0,79.0,61.0
S1099,Muskan Tiwari,Female,CSE,Hostel,85.6,,7.2,No,46.0,63.0,57.0
S1155,Khushi Pandey,Female,CSE,Hostel,76.5,4.8,5.6,No,48.0,46.0,44.0
S1009,Rahul Bansal,Male,CSE,Day Scholar,69.1,14.5,7.4,No,64.0,58.0,43.0
S1085,Ishaan Rastogi,Male,CSE,Day Scholar,80.0,12.6,7.7,No,58.0,48.0,64.0
S1028,Vivaan Malhotra,Male,CSE,Day Scholar,78.3,13.8,5.3,No,61.0,66.0,66.0
S1223,Riya Kapoor,Female,ECE,Day Scholar,74.3,19.3,6.2,No,56.0,61.0,69.0
S1067,Karan Pandey,Male,IT,Hostel,62.1,6.3,7.3,No,44.0,48.0,41.0
S1230,Khushi Verma,Female,CSE,Day Scholar,72.2,11.9,7.1,Yes,67.0,40.0,71.0
S1235,Manav Pandey,Male,ME,Hostel,100.0,5.6,7.8,No,67.0,56.0,71.0
S1178,Rohan Verma,Male,ME,Hostel,79.1,18.1,5.5,No,57.0,63.0,69.0
S1109,Shreya Saxena,Female,CSE,Hostel,80.4,10.1,7.3,No,48.0,56.0,56.0
S1126,Ishaan Sharma,Male,IT,Day Scholar,86.1,,5.4,Yes,60.0,53.0,54.0
S1047,Tanmay Sharma,Male,IT,Hostel,54.6,12.0,5.9,No,45.0,53.0,63.0
S1167,Anjali Verma,Female,ECE,Day Scholar,79.5,10.1,6.9,No,53.0,60.0,62.0
S1231,Sakshi Srivastava,Female,CSE,Day Scholar,55.9,5.7,6.8,yes,46.0,40.0,45.0
S1149,Anjali Gupta,Female,IT,Day Scholar,,8.0,6.8,No,67.0,61.0,51.0
S1082,Meera Joshi,Female,IT,Day Scholar,73.0,11.7,7.0,No,50.0,53.0,60.0
S1129,Muskan Verma,Female,CSE,Day Scholar,73.7,17.2,7.9,No,45.0,57.0,56.0
S1032,Manav Joshi,Male,ME,Hostel,52.3,11.3,6.8,No,47.0,38.0,50.0
S1250,Mohit Rastogi,Male,IT,Day Scholar,84.5,13.8,6.6,No,65.0,61.0,44.0
S1087,Neha Bansal,Female,ECE,Hostel,75.9,13.7,7.5,No,52.0,63.0,56.0
S1014,Anjali Singh,Female,ECE,hostel,79.0,6.6,6.4,No,53.0,48.0,66.0
S1010,Varun Rastogi,Male,IT,Hostel,85.9,14.4,5.5,Yes,74.0,62.0,61.0
S1019,Siddharth Bansal,Male,ECE,Day Scholar,57.7,10.9,7.7,Yes,48.0,43.0,60.0
S1096,Tanya Pandey,Female,ECE,Hostel,86.3,14.5,5.6,No,58.0,55.0,82.0
S1214,Meera Verma,Female,ECE,Hostel,76.8,8.0,6.1,No,53.0,46.0,71.0
S1240,Pooja Mishra,Female,CSE,Hostel,80.2,15.5,7.4,Yes,52.0,54.0,69.0
S1116,Sneha Kapoor,Female,ME,Day Scholar,75.0,21.1,8.6,Yes,76.0,66.0,77.0
S1015,Sneha Malhotra,Female,IT,Day Scholar,80.1,8.8,6.3,No,52.0,62.0,46.0
S1041,Pooja Verma,Female,IT,Hostel,60.9,17.5,6.6,No,54.0,57.0,62.0
S1046,Karan Sharma,Male,ECE,Day Scholar,45.0,11.6,7.1,Yes,47.0,35.0,55.0
S1203,Rohan Saxena,Male,IT,Day Scholar,64.7,15.4,6.6,Yes,54.0,50.0,71.0
S1033,Nikhil Singh,Male,CSE,Day Scholar,51.5,10.9,7.2,Yes,59.0,34.0,55.0
S1078,Muskan Malhotra,Female,ECE,Day Scholar,71.2,7.4,6.6,Yes,50.0,43.0,48.0
S1246,Nikhil Tiwari,Male,ME,Day Scholar,82.8,22.8,5.8,No,80.0,63.0,81.0
S1122,Manav Saxena,Male,ECE,Hostel,74.6,1.4,7.8,Yes,36.0,40.0,42.0
S1239,Diya Kapoor,Female,CSE,Day Scholar,73.2,7.7,6.6,No,48.0,53.0,38.0
S1029,Meera Rastogi,Female,CSE,Hostel,62.9,12.0,7.6,Yes,53.0,50.0,58.0
S1064,Aryan Jain,Male,ECE,Day Scholar,72.3,,7.1,No,55.0,54.0,60.0
S1083,Aarav Srivastava,Male,ECE,Day Scholar,97.6,5.6,5.9,No,38.0,,58.0
S1151,Aarav Malhotra,Male,ECE,Hostel,90.8,9.9,5.7,No,43.0,66.0,69.0
S1200,Rahul Pandey,Male,CSE,Day Scholar,83.4,10.4,7.0,No,56.0,73.0,69.0
S1016,Aditya Saxena,Male,CSE,Day Scholar,51.8,17.2,6.4,No,50.0,57.0,49.0
S1042,Dev Chauhan,Male,ECE,Hostel,61.9,11.3,5.0,No,51.0,32.0,46.0
S1059,Neha Jain,male,CSE,Day Scholar,76.2,15.9,6.5,No,69.0,69.0,69.0
S1135,Manav Jain,Male,IT,Hostel,70.1,9.7,5.3,No,44.0,53.0,59.0
S1194,Ananya Rastogi,Female,ECE,Hostel,86.8,6.7,7.5,No,60.0,48.0,62.0
S1037,Karan Bansal,Male,CSE,Day Scholar,74.7,3.6,8.3,No,53.0,31.0,58.0
S1206,Simran Chauhan,Female,CSE,Day Scholar,63.5,7.4,7.4,No,39.0,47.0,45.0
S1171,Muskan Pandey,Female,CSE,hostel,55.1,10.2,4.8,Yes,39.0,51.0,52.0
S1228,Arjun Rastogi,Male,IT,Day Scholar,72.6,9.0,6.8,No,30.0,48.0,57.0
S1052,Karan Tiwari,Male,CSE,Hostel,71.3,12.1,4.9,No,38.0,51.0,50.0
S1213,Aditi Pandey,Female,CSE,Hostel,92.7,2.1,6.2,No,30.0,54.0,48.0
S1115,Pooja Kapoor,Female,CSE,Day Scholar,100.0,8.5,6.9,No,59.0,48.0,65.0
S1175,Sakshi Joshi,Female,CSE,Hostel,71.7,14.6,8.1,No,62.0,51.0,58.0
S1102,Nisha Yadav,Female,ECE,Hostel,78.5,19.7,6.7,Yes,55.0,65.0,73.0
S1086,Khushi Tiwari,male,CSE,Day Scholar,72.2,13.5,7.5,No,47.0,63.0,64.0
S1098,Ananya Gupta,Female,CSE,Hostel,77.5,18.1,6.1,Yes,59.0,55.0,76.0
S1227,Aditi Tiwari,Female,CSE,Day Scholar,74.1,16.8,5.2,Yes,59.0,62.0,64.0
S1229,Karan Rastogi,Male,ECE,Day Scholar,61.6,9.8,6.6,No,53.0,36.0,63.0
S1139,Priya Saxena,Female,ECE,Hostel,,7.5,7.4,Yes,46.0,43.0,42.0
S1092,Ananya Pandey,Female,ECE,Day Scholar,78.7,12.0,7.0,No,48.0,44.0,43.0
S1148,Diya Mishra,Female,IT,Hostel,85.4,17.1,,No,58.0,67.0,77.0
S1237,Pooja Saxena,Female,IT,Day Scholar,92.2,15.0,7.1,No,59.0,55.0,75.0
S1211,Simran Verma,Female,ECE,Day Scholar,100.0,15.0,6.4,No,60.0,91.0,65.0
S1174,Dev Bansal,Male,ME,Day Scholar,75.6,8.7,6.1,No,38.0,51.0,60.0
S1038,Nikhil Joshi,Male,IT,Hostel,77.7,8.4,8.0,Yes,43.0,62.0,59.0
S1165,Anjali Agarwal,Female,IT,Day Scholar,70.5,11.8,6.8,No,72.0,43.0,58.0
S1199,Manav Tiwari,Male,CSE,Day Scholar,86.7,10.2,,No,54.0,62.0,70.0
S1187,Rahul Srivastava,Male,IT,Hostel,87.5,11.1,7.4,Yes,73.0,57.0,47.0
S1105,Riya Mishra,Female,ECE,Hostel,39.8,8.3,8.3,No,59.0,32.0,43.0
S1106,Naina Yadav,Female,ECE,Hostel,79.6,17.9,6.5,Yes,49.0,59.0,68.0
S1159,Manav Bansal,Male,ECE,Day Scholar,66.2,18.1,6.2,No,51.0,53.0,61.0
S1172,Nisha Pandey,Female,IT,Hostel,62.5,11.8,5.1,Yes,43.0,69.0,62.0
S1050,Khushi Gupta,Female,CSE,Day Scholar,70.1,4.0,6.3,No,21.0,34.0,72.0
S1142,Pooja Jain,Female,IT,Hostel,79.5,13.3,7.6,Yes,55.0,54.0,56.0
S1006,Ishaan Bansal,Male,ME,Day Scholar,66.5,15.9,7.7,No,69.0,53.0,43.0
S1018,Tanmay Yadav,Male,CSE,Day Scholar,63.2,17.9,6.4,Yes,145.0,68.0,68.0
S1130,Aarav Joshi,Male,IT,Day Scholar,77.7,13.6,6.8,No,49.0,58.0,61.0
S1103,Anjali Kapoor,Female,CSE,Day Scholar,70.7,19.8,7.1,Yes,68.0,50.0,52.0
S1162,Nisha Bansal,Female,ME,Day Scholar,75.0,9.1,6.6,No,59.0,39.0,64.0
S1114,Aditya Agarwal,Male,IT,Day Scholar,69.2,9.2,9.0,No,44.0,58.0,49.0
S1117,Aditya Gupta,Male,ECE,Day Scholar,79.9,13.2,7.2,yes,58.0,56.0,61.0
S1157,Sneha Jain,Female,CSE,Day Scholar,84.9,5.8,5.6,No,57.0,71.0,64.0
S1107,Nikhil Yadav,Male,IT,Hostel,35.0,7.8,6.2,Yes,42.0,60.0,38.0
S1163,Vivaan Saxena,Male,CSE,Hostel,79.1,9.5,7.3,Yes,61.0,67.0,50.0
S1065,Sneha Saxena,Female,ECE,Day Scholar,59.7,17.9,7.5,No,39.0,58.0,52.0
S1027,Diya Jain,male,IT,Day Scholar,74.4,8.0,7.8,No,41.0,52.0,60.0
S1140,Diya Yadav,Female,CSE,Hostel,77.8,7.3,6.9,No,46.0,51.0,42.0
S1185,Kabir Mishra,Male, cse ,Day Scholar,75.4,18.5,6.6,No,62.0,,82.0
S1224,Arjun Srivastava,male,ME,Day Scholar,62.1,11.2,5.6,Yes,58.0,63.0,54.0
S1118,Aryan Gupta,Male,ECE,Day Scholar,94.4,15.5,7.8,No,57.0,66.0,74.0
S1146,Manav Gupta,Male,CSE,Day Scholar,71.4,14.5,7.1,Yes,51.0,53.0,55.0
S1238,Arjun Kapoor,Male,ECE,Day Scholar,77.9,4.5,8.0,No,52.0,51.0,42.0
S1030,Isha Saxena,Female,ECE,Day Scholar,83.7,9.0,7.2,No,59.0,48.0,50.0
S1080,Arjun Jain,Male,ECE,Hostel,76.4,11.5,7.2,Yes,58.0,41.0,59.0
S1141,Isha Rastogi,Female,IT,Hostel,83.8,12.3,7.1,Yes,47.0,59.0,79.0
S1132,Pranav Singh,male,ME,Hostel,63.0,8.4,6.6,Yes,59.0,39.0,44.0
S1040,Aryan Agarwal,Male,ECE,Hostel,61.7,9.3,6.5,Yes,51.0,48.0,51.0
S1169,Anjali Mishra,Female,CSE,Day Scholar,92.3,12.4,7.3,Yes,55.0,80.0,
S1055,Pooja Joshi,Female,ECE,Day Scholar,64.7,10.5,6.6,No,58.0,54.0,61.0
S1044,Manav Malhotra,Male,CSE,Hostel,89.9,12.2,8.0,Yes,46.0,54.0,69.0
S1179,Naina Verma,Female,ME,Day Scholar,78.8,17.8,4.1,No,61.0,55.0,55.0
S1049,Naina Jain,Female, cse ,Hostel,68.3,2.7,8.3,No,45.0,35.0,42.0
S1191,Naina Malhotra,Female,ME,Day Scholar,80.8,15.2,5.6,No,49.0,56.0,65.0
S1060,Aditi Malhotra,Female,CSE,Hostel,73.2,13.1,6.6,No,58.0,61.0,51.0
S1144,Kabir Rastogi,Male,IT,Hostel,58.1,12.1,7.7,Yes,39.0,56.0,56.0
S1054,Kabir Pandey,Male,ECE,Hostel,66.9,13.9,7.4,Yes,55.0,44.0,53.0
S1021,Rahul Yadav,Male,ECE,Day Scholar,76.0,11.0,7.7,No,53.0,63.0,59.0
S1125,Neha Malhotra,Female,CSE,Day Scholar,66.1,3.0,5.4,No,35.0,41.0,
S1245,Priya Pandey,Female,ECE,Hostel,56.5,14.6,7.7,Yes,52.0,52.0,67.0
S1241,Anjali Joshi,Female,ECE,Day Scholar,75.7,13.1,6.1,No,58.0,58.0,59.0
S1166,Priya Singh,Female,IT,Day Scholar,84.1,19.0,8.8,Yes,58.0,80.0,82.0
S1077,Tanya Sharma,Female,CSE,Day Scholar,,10.5,6.8,Yes,42.0,37.0,48.0
S1208,Ananya Malhotra,Female, cse ,Hostel,60.1,12.2,5.2,No,48.0,57.0,49.0
S1108,Shreya Tiwari,Female,CSE,Day Scholar,54.3,6.7,6.5,Yes,35.0,47.0,45.0
S1162,Nisha Bansal,Female,ME,Day Scholar,75.0,9.1,6.6,No,59.0,39.0,64.0
S1013,Kavya Bansal,Female,ECE,Day Scholar,75.2,14.8,7.3,No,68.0,52.0,56.0
S1017,Sneha Rastogi,Female,IT,Hostel,83.0,15.4,6.5,No,55.0,48.0,66.0
S1048,Riya Jain,Female,CSE,Hostel,65.0,,6.1,No,44.0,69.0,61.0
S1183,Riya Yadav,Female,IT,Day Scholar,70.6,12.1,7.9,Yes,58.0,44.0,33.0
S1120,Varun Yadav,Male,ECE,Day Scholar,63.0,11.3,5.9,No,54.0,48.0,65.0
S1022,Rohan Gupta,Male,IT,Day Scholar,66.7,12.4,6.4,Yes,49.0,59.0,64.0
S1147,Rahul Rastogi,Male,CSE,Hostel,69.2,18.2,7.0,No,59.0,48.0,60.0
S1204,Rohan Joshi,Male,CSE,Day Scholar,68.5,5.7,5.0,No,41.0,52.0,72.0
S1131,Isha Gupta,Female,CSE,Day Scholar,82.4,15.1,4.5,Yes,80.0,80.0,60.0
S1198,Khushi Singh,Female,ME,Day Scholar,63.5,12.7,6.5,Yes,39.0,64.0,63.0
S1233,Sneha Verma,Female,ME,Hostel,77.7,15.9,5.5,No,64.0,48.0,67.0
S1074,Aditya Malhotra,Male,CSE,Hostel,92.2,21.8,9.0,No,71.0,,99.0
S1031,Pooja Rastogi,Female,ECE,Day Scholar,75.8,11.1,5.2,Yes,67.0,47.0,62.0
S1218,Pranav Malhotra,Male,CSE,Hostel,73.2,7.3,6.4,No,29.0,,59.0
S1034,Arjun Joshi,Male,IT,Hostel,74.9,19.1,7.4,No,39.0,69.0,85.0
S1181,Pooja Yadav,Female,ME,Hostel,82.4,13.5,6.6,Yes,76.0,62.0,74.0
S1209,Dev Malhotra,Male,CSE,Hostel,72.6,3.7,6.1,No,47.0,34.0,40.0
S1156,Simran Saxena,Female,ECE,Day Scholar,76.6,11.3,7.5,Yes,52.0,54.0,61.0
S1226,Harsh Gupta,Male,ME,Day Scholar,92.3,6.1,7.5,Yes,60.0,62.0,
S1124,Rohan Srivastava,Male,IT,Hostel,71.9,4.8,5.5,No,39.0,48.0,49.0
S1068,Siddharth Jain,Male,CSE,Day Scholar,66.0,7.3,6.4,Yes,70.0,44.0,49.0
S1164,Tanmay Srivastava,Male,CSE,Hostel,83.0,15.1,4.7,No,74.0,60.0,66.0
S1101,Aditi Rastogi,Female,ME,Hostel,89.4,15.0,8.2,Yes,60.0,47.0,70.0
S1150,Sakshi Gupta,Female,CSE,Hostel,96.5,18.7,6.2,Yes,75.0,75.0,63.0
S1195,Manav Sharma,Male,CSE,Day Scholar,68.3,11.9,5.1,Yes,40.0,66.0,80.0
S1249,Rohan Singh,Male,IT,Day Scholar,75.1,13.7,5.4,No,55.0,62.0,58.0
S1232,Arjun Gupta,Male,IT,Day Scholar,64.0,12.9,5.3,No,42.0,53.0,34.0
S1062,Siddharth Pandey,Male,IT,Hostel,67.5,14.8,6.2,No,51.0,61.0,61.0
S1036,Aryan Mishra,Male,CSE,Day Scholar,73.8,11.0,7.1,Yes,59.0,70.0,64.0
S1011,Priya Agarwal,Female,IT,Day Scholar,75.4,14.1,7.2,Yes,46.0,67.0,71.0
S1244,Aarav Yadav,male,IT,Hostel,58.2,13.9,4.5,Yes,56.0,55.0,84.0
S1072,Mohit Verma,Male,IT,Day Scholar,86.8,8.1,8.2,No,63.0,65.0,62.0
S1250,Mohit Rastogi,Male,IT,Day Scholar,84.5,13.8,6.6,No,65.0,61.0,44.0
S1236,Naina Rastogi,Female,ECE,Day Scholar,68.0,11.5,4.6,No,42.0,33.0,57.0
S1094,Kavya Singh,Female,IT,Hostel,61.2,10.7,6.4,No,42.0,49.0,51.0
S1160,Harsh Jain,Male,ECE,Hostel,70.0,8.8,8.1,No,42.0,54.0,59.0
S1119,Tanmay Agarwal,male,IT,Hostel,76.6,10.0,6.1,Yes,34.0,51.0,69.0
S1145,Simran Srivastava,Female,IT,Day Scholar,79.0,14.3,5.4,yes,55.0,52.0,61.0
S1215,Neha Mishra,Female,ECE,Hostel,79.9,16.9,7.6,No,60.0,45.0,65.0
S1002,Aditi Sharma,Female,IT,Day Scholar,76.6,9.4,5.8,No,54.0,41.0,56.0
S1234,Aditi Yadav,Female,IT,Day Scholar,90.8,13.2,6.8,No,67.0,60.0,44.0
S1084,Arjun Mishra,Male,CSE,Day Scholar,73.0,9.0,7.6,No,58.0,46.0,58.0
S1243,Riya Pandey,Female,CSE,Hostel,70.1,19.4,5.2,No,60.0,68.0,
S1196,Karan Malhotra,Male,CSE,Hostel,84.7,20.0,5.9,Yes,71.0,61.0,82.0
S1088,Kabir Bansal,Male,IT,Day Scholar,75.1,6.7,6.5,No,33.0,33.0,71.0
S1138,Kabir Tiwari,Male,IT,Hostel,80.6,9.8,7.2,Yes,55.0,59.0,65.0
S1005,Ishaan Singh,Male,CSE,Day Scholar,85.2,14.0,6.1,No,60.0,50.0,56.0
S1039,Nisha Mishra,Female, cse ,Day Scholar,86.4,10.9,4.0,Yes,51.0,74.0,85.0
S1207,Tanmay Bansal,Male,CSE,Day Scholar,80.3,8.1,7.2,No,46.0,42.0,52.0
S1202,Nisha Kapoor,Female,ME,Day Scholar,65.9,13.0,8.2,No,34.0,41.0,67.0
S1221,Yash Singh,Male,CSE,Hostel,88.6,13.6,5.7,No,,64.0,64.0
S1035,Kavya Malhotra,Female,IT,Day Scholar,72.3,11.6,7.0,No,66.0,70.0,52.0
S1193,Neha Kapoor,Female,CSE,Hostel,75.0,13.0,5.8,No,65.0,62.0,62.0
S1190,Naina Tiwari,Female, cse ,Day Scholar,69.0,3.9,7.8,Yes,24.0,54.0,61.0
S1113,Sneha Srivastava,Female,CSE,Hostel,91.4,9.3,6.7,Yes,76.0,51.0,61.0
S1182,Nikhil Malhotra,Male,ME,Hostel,49.7,9.5,6.4,No,52.0,32.0,49.0
S1184,Khushi Jain,Female,CSE,Day Scholar,48.8,9.6,8.0,No,30.0,49.0,36.0
S1153,Yash Malhotra,Male,CSE,Day Scholar,79.2,12.3,5.7,Yes,51.0,64.0,58.0
S1057,Ananya Yadav,Female,IT,Day Scholar,56.6,16.1,5.4,Yes,49.0,48.0,62.0
S1070,Pooja Pandey,Female,IT,Hostel,81.7,11.0,5.2,No,53.0,61.0,70.0
S1192,Kavya Gupta,Female,CSE,Day Scholar,68.7,12.2,8.8,No,57.0,54.0,60.0
S1045,Naina Mishra,Female,ECE,Day Scholar,69.8,12.1,5.7,No,60.0,55.0,64.0
S1189,Aryan Tiwari,Male,ECE,Day Scholar,77.4,3.3,7.1,No,45.0,41.0,49.0
S1020,Kavya Agarwal,Female,ECE,Hostel,74.3,7.4,5.1,No,47.0,62.0,61.0
S1007,Tanya Rastogi,Female,ECE,Day Scholar,86.6,17.0,6.7,No,52.0,70.0,81.0
S1056,Ananya Saxena,Female,ECE,Day Scholar,80.7,16.1,,Yes,61.0,61.0,65.0
S1076,Rohan Mishra,Male,CSE,Hostel,91.4,10.4,7.8,Yes,62.0,59.0,58.0
S1001,Aarav Kapoor,Male,ECE,Hostel,73.9,12.7,6.3,No,47.0,42.0,66.0
S1173,Khushi Rastogi,Female,CSE,Hostel,62.7,11.5,7.0,Yes,51.0,56.0,65.0
S1136,Muskan Mishra,Female,CSE,Hostel,58.0,14.2,6.4,No,51.0,66.0,75.0
S1128,Yash Rastogi,Male,CSE,Day Scholar,71.6,11.8,6.9,Yes,65.0,55.0,61.0
S1177,Sneha Singh,Female,IT,Hostel,82.4,14.4,4.9,Yes,87.0,68.0,62.0
S1069,Meera Agarwal,Female,CSE,Hostel,79.8,10.5,9.0,Yes,47.0,63.0,53.0
S1231,Sakshi Srivastava,Female,CSE,Day Scholar,55.9,5.7,6.8,yes,46.0,40.0,45.0
S1168,Sakshi Malhotra,Female,ECE,Day Scholar,60.2,16.0,6.9,Yes,46.0,47.0,40.0
S1137,Kabir Sharma,Male,ME,Day Scholar,80.3,14.3,8.5,Yes,61.0,67.0,68.0
S1219,Yash Agarwal,Male,ME,Day Scholar,92.4,15.2,6.9,No,44.0,79.0,69.0
S1043,Pranav Yadav,Male,ECE,Day Scholar,78.5,17.2,6.9,No,69.0,64.0,61.0
S1111,Isha Malhotra,Female,IT,Hostel,61.1,11.6,6.4,No,27.0,44.0,59.0
S1090,Simran Yadav,Female,IT,Day Scholar,89.0,8.8,5.9,No,49.0,48.0,42.0
S1073,Pranav Jain,Male,ME,Day Scholar,88.9,8.6,5.9,No,47.0,28.0,69.0
S1024,Priya Chauhan,Female,ME,Day Scholar,58.9,16.5,5.7,No,79.0,53.0,58.0
S1143,Simran Singh,Female,ME,Day Scholar,82.6,8.8,6.0,Yes,64.0,55.0,51.0
S1186,Nisha Joshi,Female,ECE,Hostel,74.9,13.5,4.7,Yes,48.0,62.0,58.0
S1093,Kabir Gupta,Male,IT,Day Scholar,82.1,6.5,6.7,Yes,59.0,69.0,58.0
S1104,Siddharth Malhotra,Male,ME,Day Scholar,79.0,7.1,9.0,No,44.0,44.0,52.0
S1152,Priya Verma,Female,ME,Day Scholar,73.7,16.2,,No,63.0,60.0,66.0
S1212,Kavya Chauhan,Female,IT,Hostel,58.5,11.0,5.4,No,47.0,52.0,55.0
S1068,Siddharth Jain,Male,CSE,Day Scholar,66.0,7.3,6.4,Yes,70.0,44.0,49.0
S1247,Sakshi Bansal,Female,ECE,Day Scholar,71.7,11.1,6.3,Yes,47.0,61.0,67.0
S1026,Dev Sharma,Male,CSE,Hostel,68.0,13.4,8.0,No,55.0,51.0,59.0
S1197,Pooja Srivastava,Female,IT,Day Scholar,83.1,15.3,8.3,No,68.0,63.0,50.0
S1176,Kavya Jain,Female,CSE,Day Scholar,71.0,14.6,6.5,yes,46.0,62.0,43.0
"""

raw_path = BASE / "data/raw/student_performance.csv"
if not raw_path.exists():          # never overwrite an existing raw file
    raw_path.write_text(RAW_CSV.lstrip(), encoding="utf-8")
print("Raw data ready:", raw_path)


## Step 2 – Load the dataset
Read the raw CSV file. The raw file is never overwritten.

In [ ]:
df = pd.read_csv(BASE / "data/raw/student_performance.csv")
print("Shape:", df.shape)
display(df.head())

## Step 3 – Inspect the data
Inspect column names, data types, missing values, and summary statistics.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all"))

print("\nMissing values by column:")
display(df.isnull().sum())

## Step 4 – Check duplicate records
Count exact duplicate rows before removing them.

In [ ]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

## Step 5 – Remove exact duplicates
Keep the first occurrence of each fully identical row.

In [ ]:
df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)

## Step 6 – Clean text columns
Remove leading/trailing spaces and standardize capitalization (`yes` → `Yes`, ` cse ` → `CSE`).

In [ ]:
for column in ["Gender", "Residence", "Tutoring"]:
    df[column] = df[column].astype("string").str.strip().str.title()

df["Branch"] = df["Branch"].astype("string").str.strip().str.upper()
df["Student_Name"] = df["Student_Name"].astype("string").str.strip().str.title()

for column in ["Gender", "Branch", "Residence", "Tutoring"]:
    print(column, "->", df[column].unique().tolist())

## Step 7 – Correct data types
Convert numeric columns. Values that cannot be converted become missing.

In [ ]:
mark_cols = ["Maths_Marks", "Physics_Marks", "Programming_Marks"]
num_cols = ["Attendance_Pct", "Study_Hours_Per_Week", "Sleep_Hours"] + mark_cols

for column in num_cols:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.dtypes

## Step 8 – Identify invalid values
For this practice dataset, marks must be between 0 and 100 and attendance between 0 and 100. Treat values outside these ranges as invalid.

In [ ]:
invalid_marks = (df[mark_cols] > 100).sum().sum() + (df[mark_cols] < 0).sum().sum()
print("Invalid mark values found:", invalid_marks)

for column in mark_cols + ["Attendance_Pct"]:
    df.loc[(df[column] > 100) | (df[column] < 0), column] = np.nan

print("\nMissing values after conversion and validation:")
display(df.isnull().sum())

## Step 9 – Handle missing values
Student ID, name, gender, branch, residence and tutoring are complete. Missing numeric values are filled with the column median, which is not affected by extreme values.

In [ ]:
missing_before = int(df[num_cols].isnull().sum().sum())
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
missing_after = int(df.isnull().sum().sum())

print("Missing numeric values filled:", missing_before)
print("Missing values left:", missing_after)
print("Shape after cleaning:", df.shape)

## Step 10 – Create new columns
Average and total marks, result band, attendance group and an at-risk flag (average marks below 50).

In [ ]:
df["Average_Marks"] = df[mark_cols].mean(axis=1).round(1)
df["Total_Marks"] = df[mark_cols].sum(axis=1)

def band(marks):
    if marks >= 70: return "Distinction"
    if marks >= 60: return "First Class"
    if marks >= 50: return "Pass"
    return "At Risk"

df["Result_Band"] = df["Average_Marks"].apply(band)
df["Attendance_Group"] = pd.cut(df["Attendance_Pct"], [0, 60, 75, 100],
                                labels=["Low (<60)", "Medium (60-75)", "High (>75)"])
df["At_Risk"] = df["Average_Marks"] < 50

display(df[["Student_ID", "Student_Name", "Average_Marks", "Total_Marks", "Result_Band", "Attendance_Group", "At_Risk"]].head())

## Step 11 – Calculate key metrics
Answer: overall performance of the batch.

In [ ]:
total_students = len(df)
print("Total students:", total_students)
print("Mean average marks:", round(df["Average_Marks"].mean(), 1))
print("Median average marks:", df["Average_Marks"].median())
print("Highest / lowest:", df["Average_Marks"].max(), "/", df["Average_Marks"].min())

print("\nResult band counts:")
display(df["Result_Band"].value_counts())

## Step 12 – Analyze performance by group
`groupby()` groups students by a category, and `mean()` gives the average marks of each group. Answer: does performance differ by branch, gender, residence or tutoring?

In [ ]:
for column in ["Branch", "Gender", "Residence", "Tutoring"]:
    print(column)
    display(df.groupby(column)["Average_Marks"].mean().round(1).sort_values(ascending=False))

## Step 13 – Check attendance and study hours
Correlation shows association, not cause. Answer: do attendance and study hours affect marks?

In [ ]:
corr = df[["Attendance_Pct", "Study_Hours_Per_Week", "Sleep_Hours", "Average_Marks"]].corr().round(2)
display(corr["Average_Marks"])

## Step 14 – Identify at-risk students
Students with average marks below 50 need early mentor support.

In [ ]:
at_risk = df[df["At_Risk"]].sort_values("Average_Marks")
print(f"{len(at_risk)} students at risk ({len(at_risk) / len(df):.1%} of the batch)")

print("\nAt-risk rate by attendance group:")
display(df.groupby("Attendance_Group", observed=True)["At_Risk"].mean().round(3))

display(at_risk[["Student_ID", "Student_Name", "Branch", "Attendance_Pct", "Study_Hours_Per_Week", "Average_Marks"]].head(15))

## Step 15 – Visualize the distribution of marks
A histogram shows how average marks are spread across the batch.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["Average_Marks"], bins=20, kde=True, color="steelblue")
plt.axvline(50, color="red", linestyle="--", label="Risk cut-off (50)")
plt.title("Distribution of Average Marks")
plt.legend()
plt.tight_layout()
plt.savefig(BASE / "images/average_marks_histogram.png", dpi=150)
plt.show()

## Step 16 – Visualize attendance and study hours vs marks
Scatter plots with a trend line show the relationship.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.regplot(data=df, x="Attendance_Pct", y="Average_Marks", ax=axes[0],
            scatter_kws={"alpha": 0.5}, line_kws={"color": "red"})
axes[0].set_title("Attendance vs Average Marks")
sns.regplot(data=df, x="Study_Hours_Per_Week", y="Average_Marks", ax=axes[1],
            scatter_kws={"alpha": 0.5}, line_kws={"color": "red"})
axes[1].set_title("Study Hours vs Average Marks")
plt.tight_layout()
plt.savefig(BASE / "images/attendance_study_scatter.png", dpi=150)
plt.show()

## Step 17 – Visualize correlations
A heatmap shows how all numeric columns relate to each other.

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[["Attendance_Pct", "Study_Hours_Per_Week", "Sleep_Hours",
                "Maths_Marks", "Physics_Marks", "Programming_Marks", "Average_Marks"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(BASE / "images/correlation_heatmap.png", dpi=150)
plt.show()

## Step 18 – Visualize marks by group
Bar charts make group comparisons easier.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, ["Branch", "Gender", "Residence", "Tutoring"]):
    order = df.groupby(col)["Average_Marks"].mean().sort_values(ascending=False)
    ax.bar(order.index, order.values, color=sns.color_palette("viridis", len(order)))
    ax.set_title(f"Average Marks by {col}")
    ax.set_ylabel("Average Marks")
    ax.set_ylim(0, 80)
    for i, v in enumerate(order.values):
        ax.text(i, v + 1, f"{v:.1f}", ha="center")
plt.tight_layout()
plt.savefig(BASE / "images/marks_by_group.png", dpi=150)
plt.show()

## Step 19 – Extension tasks
Top 10 students by total marks, a sleep-hours insight, and a donut chart of Result_Band.

In [ ]:
top10 = df.sort_values("Total_Marks", ascending=False).head(10)
display(top10[["Student_ID", "Student_Name", "Branch", "Total_Marks", "Average_Marks"]].reset_index(drop=True))

print("Correlation (Sleep_Hours vs Average_Marks):", round(df["Sleep_Hours"].corr(df["Average_Marks"]), 2))
sleep_group = pd.cut(df["Sleep_Hours"], [0, 6, 7.5, 12], labels=["<6 h", "6-7.5 h", ">7.5 h"])
display(df.groupby(sleep_group, observed=True)["Average_Marks"].mean().round(1))

counts = df["Result_Band"].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=90, pctdistance=0.78,
        wedgeprops={"width": 0.45}, colors=sns.color_palette("Set2"))
plt.title("Result Band Distribution")
plt.tight_layout()
plt.savefig(BASE / "images/result_band_donut.png", dpi=150)
plt.show()

## Step 20 – Export the cleaned data
Save the cleaned data (CSV and Excel) in a separate folder so the raw file is never overwritten.

In [ ]:
df.to_csv(BASE / "data/processed/student_performance_clean.csv", index=False)
df.to_excel(BASE / "data/processed/student_performance_clean.xlsx", index=False)
print("Export complete!")

## Step 21 – Student interpretation
Write 4-5 observations in your own words:

1. What is the overall performance of the batch (mean, spread, result bands)?
2. Do attendance and study hours affect marks? How strong is the relationship?
3. Does performance differ by branch, gender, residence or tutoring?
4. How many students are at risk, and what do they have in common?
5. What data-quality problems were found, and how were they handled?

**Important:** The cleaning choices in this notebook are teaching assumptions. In a real project, confirm business rules with the data owner. The data is synthetic, and correlation is not causation.